## fine-tuning CamemBERT v5 — scoring sémantique AISCA

Cette version corrige les principaux problèmes identifiés en v3/v4 :
le modèle avait tendance à rapprocher des blocs sans rapport (ML et Juridique
autour de 0.45), ce qui cassait complètement le scoring sur certains profils.
La solution choisie : des paires négatives explicites pour ancrer les blocs éloignés,
et des paires voisines avec un label intermédiaire (0.4) pour éviter que le modèle
ne binarise trop fort similaire/différent.

Autre décision importante : B00 (soft skills transversaux) est exclu des paires positives.
Si on le traite comme les autres blocs, ses compétences forment un cluster central
dans l'espace vectoriel qui perturbe tous les calculs de similarité en aval.
On le garde uniquement en paires voisines avec tous les blocs.

Note : migration vers SentenceTransformerTrainer (sentence-transformers >= 3.0)
pour corriger un TypeError avec l'ancienne API. Tourne en fp16 sur GPU, ~3 min.


In [1]:
import json
import random
import os
from itertools import combinations
from pathlib import Path

import torch
from datasets import Dataset
from sentence_transformers import (
    SentenceTransformer,
    SentenceTransformerTrainer,
    SentenceTransformerTrainingArguments,
    losses,
    evaluation,
    util,
)

# verif env au demarrage, utile quand on bosse sous WSL et que le GPU
# n'est pas toujours visible selon la config CUDA
print(f"Repertoire courant : {os.getcwd()}")
print(f"CUDA disponible    : {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU                : {torch.cuda.get_device_name(0)}")
    print(f"VRAM               : {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} Go")


2026-03-17 12:02:23.300957: I tensorflow/core/util/port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
2026-03-17 12:02:23.366133: I tensorflow/core/platform/cpu_feature_guard.cc:210] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 AVX512F AVX512_VNNI FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
2026-03-17 12:02:37.766452: I tensorflow/core/util/port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.


Répertoire courant : /mnt/d/WSL_fichier/Environnement/envs/tf_gpu
CUDA disponible    : True
GPU                : NVIDIA GeForce RTX 3070 Laptop GPU
VRAM               : 8.6 Go


In [2]:
# Blocs qui se recoupent dans la realite metier — un profil Data fait souvent
# aussi du ML ou du Dev, et inversement. On leur donne un label intermediaire
# plutot que de les pousser comme des blocs etrangers l'un a l'autre.
BLOCS_VOISINS = [
    ("B08", "B09"),   # Data Analysis <-> ML, overlap frequent
    ("B09", "B10"),   # ML <-> Dev, pareil
    ("B08", "B10"),   # Data <-> Dev
    ("B04", "B02"),   # Finance <-> Business
    ("B05", "B06"),   # Communication <-> Design
    ("B05", "B07"),   # Communication <-> Digital
    # B00 = soft skills, voisin de tous les blocs sans exception.
    # Exclure B00 des paires positives est une decision deliberee :
    # ses competences sont generiques par nature, les forcer similaires
    # entre elles creerait un cluster central qui biaiserait tout l'espace.
    ("B00", "B01"),
    ("B00", "B02"),
    ("B00", "B03"),
    ("B00", "B04"),
    ("B00", "B05"),
    ("B00", "B06"),
    ("B00", "B07"),
    ("B00", "B08"),
    ("B00", "B09"),
    ("B00", "B10"),
    ("B00", "B11"),
]

# Paires negatives choisies manuellement sur les cas qui posaient probleme.
# En v3, ML <-> Juridique obtenait ~0.45 de similarite cosinus, ce qui est
# aberrant pour deux domaines sans rapport. Ces exemples forces ancrent
# correctement l'espace vectoriel sur ces cas limites.
NEGATIFS_EXPLICITES = [
    ("Je maitrise TensorFlow PyTorch et scikit-learn",
     "Je redige des actes juridiques et documents legaux",
     0.05),
    ("Je developpe des reseaux de neurones profonds pour le deep learning",
     "J'assure la conformite reglementaire et le respect des normes RGPD",
     0.05),
    ("Je maitrise le droit des affaires et le droit commercial",
     "Je construis des modeles predictifs pour la classification",
     0.05),
    ("Je cree des designs graphiques professionnels et esthetiques",
     "Je deploie des infrastructures cloud AWS Azure GCP",
     0.05),
    ("Je gere les relations publiques et les relations avec la presse",
     "Je concois des systemes mecaniques complexes et innovants",
     0.05),
]


def build_training_pairs(json_path: str):
    with open(json_path, 'r', encoding='utf-8') as f:
        data = json.load(f)

    blocs = {}
    for bloc in data['blocs']:
        bid = bloc['id']
        blocs[bid] = [comp['texte'] for comp in bloc['competences']]

    bloc_ids = list(blocs.keys())

    # Paires positives : deux competences du meme bloc -> label 0.9
    # On utilise 0.9 et pas 1.0 car deux competences d'un meme bloc
    # ne sont pas strictement identiques, juste semantiquement proches.
    # B00 exclu pour les raisons mentionnees plus haut.
    positives = []
    for bid, textes in blocs.items():
        if bid == 'B00':
            continue
        for t1, t2 in combinations(textes, 2):
            positives.append((t1, t2, 0.9))

    # Paires voisines : blocs connexes -> label 0.4
    voisines = []
    voisins_set = set()
    for b1, b2 in BLOCS_VOISINS:
        if b1 in blocs and b2 in blocs:
            voisins_set.add((min(b1, b2), max(b1, b2)))
            for t1 in blocs[b1]:
                for t2 in blocs[b2]:
                    voisines.append((t1, t2, 0.4))

    # Paires negatives : blocs sans lien -> label 0.1
    # Les negatifs explicites passent en premier pour garantir leur presence
    # dans le jeu d'entrainement apres le shuffle. B00 exclu (voisin de tout).
    # L'objectif de target est d'equilibrer le jeu entre les trois categories.
    negatives = list(NEGATIFS_EXPLICITES)
    target = len(positives) + len(voisines)
    attempts = 0
    while len(negatives) < target and attempts < target * 20:
        attempts += 1
        b1, b2 = random.sample(bloc_ids, 2)
        if b1 == 'B00' or b2 == 'B00':
            continue
        if (min(b1, b2), max(b1, b2)) in voisins_set:
            continue
        t1 = random.choice(blocs[b1])
        t2 = random.choice(blocs[b2])
        negatives.append((t1, t2, 0.1))

    all_pairs = positives + voisines + negatives
    random.shuffle(all_pairs)

    split = int(len(all_pairs) * 0.8)
    train_data = all_pairs[:split]
    eval_data  = all_pairs[split:]

    print(f"Paires positives : {len(positives)} (B00 exclu)")
    print(f"Paires voisines  : {len(voisines)}")
    print(f"Paires negatives : {len(negatives)} (dont {len(NEGATIFS_EXPLICITES)} explicites)")
    print(f"Total            : {len(all_pairs)}")
    print(f"Train : {len(train_data)} | Eval : {len(eval_data)}")

    train_dataset = Dataset.from_dict({
        'sentence1': [x[0] for x in train_data],
        'sentence2': [x[1] for x in train_data],
        'score':     [x[2] for x in train_data],
    })
    eval_dataset = Dataset.from_dict({
        'sentence1': [x[0] for x in eval_data],
        'sentence2': [x[1] for x in eval_data],
        'score':     [x[2] for x in eval_data],
    })

    return train_dataset, eval_dataset


In [3]:
def finetune(json_path: str, output_path: str = "scoring-camembert-v5"):

    train_dataset, eval_dataset = build_training_pairs(json_path)

    # On repart systematiquement de camembert-base plutot que d'un checkpoint
    # existant pour eviter d'accumuler le biais des versions precedentes.
    # Repartir de v4 en v4->v5 introduisait de l'overfitting sur les anciens patterns.
    model = SentenceTransformer('camembert-base')

    device = "cuda" if torch.cuda.is_available() else "cpu"
    model = model.to(device)
    print(f"Entrainement sur : {device.upper()}")

    train_loss = losses.CosineSimilarityLoss(model)

    evaluator = evaluation.EmbeddingSimilarityEvaluator(
        sentences1=eval_dataset['sentence1'],
        sentences2=eval_dataset['sentence2'],
        scores=eval_dataset['score'],
        name='eval-referentiel',
    )

    # 64 tient en VRAM sur RTX 3070 (8.6 Go) sans OOM, avec une bonne vitesse.
    # En dessous de 32 le gradient devient trop bruité sur ce corpus.
    batch_size = 64 if device == "cuda" else 16

    args = SentenceTransformerTrainingArguments(
        output_dir=output_path,
        num_train_epochs=5,              # 3 donnaient un plateau trop tot
        per_device_train_batch_size=batch_size,
        per_device_eval_batch_size=batch_size,
        warmup_ratio=0.1,                # standard pour eviter le spike de loss initial
        eval_strategy='steps',
        eval_steps=50,                   # eval frequente pour sauvegarder le meilleur checkpoint
        save_strategy='best',
        load_best_model_at_end=True,     # critique : on veut le meilleur Spearman, pas le dernier
        metric_for_best_model='eval-referentiel_spearman_cosine',
        logging_steps=20,
        fp16=device == "cuda",           # reduit le temps d'entrainement d'environ 50% sur GPU
        no_cuda=device == "cpu",
    )

    trainer = SentenceTransformerTrainer(
        model=model,
        args=args,
        train_dataset=train_dataset,
        eval_dataset=eval_dataset,
        loss=train_loss,
        evaluator=evaluator,
    )

    trainer.train()

    model.save(output_path)
    print(f"Modele sauvegarde dans : {output_path}")
    return model


In [4]:
# Chemins WSL — adapter si execution sur Windows natif
DATA_DIR   = Path("/mnt/d/Etude/Master cours/IA Generative/environnement/projet-aisca/data")
input_path = DATA_DIR / 'referentiel_v3.json'

print("Fichier trouve :", input_path.exists())
print("Chemin complet :", input_path)

output_path = "/mnt/d/Etude/Master cours/IA Generative/environnement/projet-aisca/src/scoring-camembert-v5"

model_new = finetune(str(input_path), output_path=output_path)


Fichier trouvé : True
Chemin complet : /mnt/d/Etude/Master cours/IA Generative/environnement/projet-aisca/data/referentiel_v3.json
Paires positives : 524 (B00 exclu)
Paires voisines  : 2299
Paires négatives : 2823 (dont 5 explicites)
Total            : 5646
Train : 4516 | Eval : 1130


No sentence-transformers model found with name camembert-base. Creating a new one with mean pooling.


Entraînement sur : CUDA


Computing widget examples:   0%|          | 0/1 [00:00<?, ?example/s]

Step,Training Loss,Validation Loss,Eval-referentiel Pearson Cosine,Eval-referentiel Spearman Cosine
50,0.046400,0.024198,0.756475,0.762809
100,0.009200,0.006679,0.937991,0.887002
150,0.004900,0.003019,0.972798,0.892998
200,0.003900,0.002648,0.979943,0.895673
250,0.003900,0.002274,0.980885,0.895643
300,0.002000,0.001983,0.982403,0.895662
350,0.002300,0.001904,0.983416,0.895742


Modèle sauvegardé dans : /mnt/d/Etude/Master cours/IA Generative/environnement/projet-aisca/src/scoring-camembert-v5


In [5]:
# Comparaison v4 vs v5 sur les paires cibles.
# Ce qu'on attend : ML<->ML et Dev<->Dev en hausse, ML<->Juridique en baisse nette.
DATA_DIR      = Path("/mnt/d/Etude/Master cours/IA Generative/environnement/projet-aisca/src")
model_path_v4 = Path("scoring-camembert-v4")
model_path_v5 = Path("scoring-camembert-v5")

model_v4 = SentenceTransformer(str(DATA_DIR / model_path_v4))
model_v5 = SentenceTransformer(str(DATA_DIR / model_path_v5))

paires = [
    ("Je maitrise TensorFlow et PyTorch",
     "Je developpe des reseaux de neurones profonds",
     "ML <-> ML"),
    ("Je developpe des API REST avec FastAPI",
     "Je gere des bases de donnees PostgreSQL",
     "Dev <-> Dev"),
    ("Je maitrise TensorFlow et PyTorch",
     "Je redige des actes juridiques et contrats",
     "ML <-> Juridique"),    # doit etre proche de 0
    ("Je programme en Python avec Pandas NumPy",
     "Je developpe des algorithmes de machine learning",
     "Data <-> ML"),
    ("Faire preuve de rigueur et de precision dans son travail",
     "Avoir une grande rigueur dans le traitement des chiffres",
     "B00 <-> Finance"),
    ("Avoir un esprit analytique et logique pour resoudre des problemes complexes",
     "Je realise des analyses exploratoires EDA",
     "B00 <-> Data"),
    ("Je gere les relations publiques et la presse",
     "Je deploie des infrastructures cloud AWS",
     "Comm <-> Dev"),        # doit rester bas
]

def scores_paires(model, paires):
    resultats = []
    for t1, t2, _ in paires:
        e1 = model.encode(t1, convert_to_tensor=True)
        e2 = model.encode(t2, convert_to_tensor=True)
        resultats.append(float(util.cos_sim(e1, e2)))
    return resultats

scores_v4 = scores_paires(model_v4, paires)
scores_v5 = scores_paires(model_v5, paires)

print(f"{'Paire':<20} {'v4':>8} {'v5':>8} {'Delta':>8}")
print("-" * 50)
for i, (_, _, label) in enumerate(paires):
    delta = scores_v5[i] - scores_v4[i]
    signe = "+" if delta > 0 else ""
    print(f"{label:<20} {scores_v4[i]:>8.3f} {scores_v5[i]:>8.3f} {signe+f'{delta:.3f}':>8}")


Paire                      v4       v5        Δ
--------------------------------------------------
ML ↔ ML                 0.797    0.807   +0.010
Dev ↔ Dev               0.850    0.886   +0.036
ML ↔ Juridique          0.118    0.093   -0.026
Data ↔ ML               0.506    0.493   -0.013
B00 ↔ Finance           0.533    0.677   +0.144
B00 ↔ Data              0.521    0.431   -0.090
Comm ↔ Dev              0.099    0.100   +0.001


In [6]:
# Benchmark Spearman sur paires independantes — non vues pendant l'entrainement.
# L'objectif est de verifier qu'on generalise correctement, pas qu'on a juste
# memorise les exemples du corpus. La correlation de Spearman mesure si le
# classement des scores du modele correspond au classement des labels humains.
paires_bench = [
    ("Je deploie des modeles ML en production avec MLOps",
     "J'optimise les hyperparametres et evalue les performances",
     0.9),
    ("Je cree des visualisations avec Tableau Power BI",
     "Je realise des analyses statistiques descriptives",
     0.7),
    ("Je programme en Python avec Pandas pour l'analyse",
     "Je developpe des reseaux de neurones avec PyTorch",
     0.4),   # Data et ML sont lies mais distincts
    ("Je maitrise la comptabilite generale en normes IFRS",
     "Je gere des communautes en ligne sur les reseaux sociaux",
     0.1),
    ("Je developpe des API REST avec Flask Django",
     "Je gere des bases de donnees PostgreSQL MongoDB",
     0.75),
    ("Je redige des communiques de presse impactants",
     "Je deploie des infrastructures cloud AWS Azure",
     0.05),
    ("Je maitrise la strategie d'entreprise et business models",
     "Je realise des analyses financieres et diagnostics",
     0.4),
    ("Je cree des designs graphiques avec Adobe Illustrator",
     "Je gere les relations publiques et situations de crise",
     0.2),
    # B00 : verification du positionnement du bloc transversal
    ("Faire preuve de rigueur et de precision dans son travail",
     "Avoir une grande rigueur dans le traitement des chiffres",
     0.7),
    ("Avoir un esprit analytique et logique",
     "Je realise des analyses exploratoires EDA",
     0.6),
    # Surveillance de regression : ML <-> Juridique etait a ~0.45 en v3
    ("Je maitrise TensorFlow PyTorch et scikit-learn",
     "Je redige des actes juridiques et documents legaux",
     0.05),
]

evaluator_bench = evaluation.EmbeddingSimilarityEvaluator(
    sentences1=[p[0] for p in paires_bench],
    sentences2=[p[1] for p in paires_bench],
    scores=    [p[2] for p in paires_bench],
    name='benchmark-independant',
)

result_v4 = evaluator_bench(model_v4)
result_v5 = evaluator_bench(model_v5)

# L'API renvoie un dict ou un float selon la version de sentence-transformers
def get_spearman(result):
    if isinstance(result, dict):
        key = 'benchmark-independant_spearman_cosine'
        return result.get(key, result.get(next(k for k in result if 'spearman' in k)))
    return result

spearman_v4 = get_spearman(result_v4)
spearman_v5 = get_spearman(result_v5)

print(f"Spearman v4 : {spearman_v4:.4f}")
print(f"Spearman v5 : {spearman_v5:.4f}")
delta = spearman_v5 - spearman_v4
signe = "+" if delta > 0 else ""
print(f"Delta       : {signe}{delta:.4f}  {'amelioration' if delta > 0 else 'regression'}")


Spearman v4 : 0.9016
Spearman v5 : 0.9565
Δ           : +0.0549  ✓ amélioration
